# Imports

In [ ]:
import sys
import pandas as pd
import os
import json
import re

# 1. Path Setup for Docker/Local consistency
current_dir = os.getcwd()
if os.path.basename(current_dir) == 'ETL':
    project_root = os.path.abspath(os.path.join(current_dir, '..'))
else:
    project_root = current_dir

if project_root not in sys.path:
    sys.path.append(project_root)

# 2. Ensure working directory is correct for data loading
if os.path.basename(os.getcwd()) != 'ETL' and os.path.exists('ETL'):
    os.chdir('ETL')

print(f"Working Directory: {os.getcwd()}")

# Configurations

In [ ]:
# Define relative paths
INPUT_FILE_EVENTS = "../data_raw/wix_events_data.json"
OUTPUT_FOLDER = "../data_temp/"
OUTPUT_FILE = os.path.join(OUTPUT_FOLDER, "events_1_intermediate.json")

# Simple check to verify the file is where we think it is
if os.path.exists(INPUT_FILE_EVENTS):
    print(f"✅ Setup complete. Input file found: {INPUT_FILE_EVENTS}")
else:
    print(f"❌ Warning: Input file NOT found at {INPUT_FILE_EVENTS}")

# Ensure the output directory exists
if os.path.exists(OUTPUT_FOLDER):
    print(f"✅ Setup complete. Output folder found: {OUTPUT_FOLDER}")
else:
    os.makedirs(OUTPUT_FOLDER)
    print(f"✅ Created Output folder: {OUTPUT_FOLDER}")

# Load raw data

In [ ]:
try:
    with open(INPUT_FILE_EVENTS, 'r', encoding='utf-8') as f:
        raw_data = json.load(f)
    
    events_df = pd.DataFrame(raw_data)
    print(f"✅ Successfully loaded {len(events_df)} records.")
    
    display(events_df.head(1)) 
    
    print("\n📑 Available columns:", *events_df.columns, sep="\n")
  
except FileNotFoundError:
    print(f"Error: The file {INPUT_FILE_EVENTS} was not found.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

# Flattening data

In [ ]:
try:
    # Use the raw_data already loaded in the 'Data Loading' section
    events_df = pd.json_normalize(raw_data)
    display(events_df.head(3)) 
    
    print("\n📑 Available flattened columns (first 10):", *events_df.columns, sep="\n")
    
except NameError:
    print("Error: 'raw_data' is not defined. Please run the 'Data Loading' cell first.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

# Transformation logic - Part_1

In [ ]:
final_events_df = events_df.assign(
    id=events_df['_id'],
    title=events_df['title'],
    location=events_df.get('location.type'),
    url=events_df['eventPageUrl'],

    # Standardize the date to string format
    date=pd.to_datetime(events_df['dateAndTimeSettings.startDate']).dt.date.astype(str),

    # Group Guests into a single nested object
    # .fillna(0) ensures we don't put 'nan' into our final dictionary
    eventGuests=events_df.fillna(0).apply(lambda row: {
        'total': int(row.get('summaries.rsvps.totalCount', 0)),
        'going': int(row.get('summaries.rsvps.yesCount', 0)),
        'notGoing': int(row.get('summaries.rsvps.noCount', 0)),
        'waitlist': int(row.get('summaries.rsvps.waitlistCount', 0)),
    }, axis=1)
)

# Reorder and keep only necessary columns
final_columns = ['id', 'title', 'date', 'location', 'eventGuests', 'url']
final_events_df = final_events_df[final_columns]

print(f"✅ Transformed {len(final_events_df)} events.")
display(final_events_df.head(3))

# Transformation logic - Part_2

In [ ]:
# Extract category names
# We use .get() and check if 'val' is a list to prevent crashes on empty/NaN rows
final_events_df['categories'] = events_df['categories.categories'].apply(
    lambda val: [item.get('name') for item in val] if isinstance(val, list) else []
)

# Final selection and Reorder
final_columns = ['id', 'title', 'date', 'location', 'categories', 'eventGuests', 'url']
# If 'categories' wasn't in the original assign, we must ensure it exists 
final_events_df = final_events_df.reindex(columns=final_columns)

print("✅ Category extraction complete.")
display(final_events_df[['title', 'categories']].head())

# Get unique categories by exploding the list directly
# This takes the list in each row and turns it into individual rows
all_categories = final_events_df['categories'].explode()

# Drop NA/Empty values and get unique set
unique_categories = all_categories.dropna().unique()

print("\n" + "="*50 + "\n")
print(f"📑 Found {len(unique_categories)} unique categories:")
print(*sorted(unique_categories), sep="\n")

# Transformation logic - Part_3

In [ ]:
# Clean categories
TAG_TO_REMOVE = 'publish'

def clean_and_join_categories(cat_list):
    if not isinstance(cat_list, list):
        return None
    
    # Filter out 'publish' and any empty/whitespace strings
    filtered_list = [c.strip() for c in cat_list if c and c.strip() != TAG_TO_REMOVE]
    
    # If the list is empty, return None
    if not filtered_list:
        return None
        
    return ", ".join(filtered_list)

# Create the new column based on the existing list column
final_events_df['categories'] = final_events_df['categories'].apply(clean_and_join_categories)

# 2. Recalculate unique categories for verification
# We dropna() because the function now returns None if a list was empty or only contained 'publish'
# Note: Since the column is now strings, not lists, we don't need .explode() 
# for unique, but we do need to split them if they have multiple categories.
unique_categories = final_events_df['categories'].dropna().str.split(', ').explode().unique()

print(f"✅ Found {len(unique_categories)} unique categories (excluding '{TAG_TO_REMOVE}'):")
print(*unique_categories, sep="\n")

print("\n" + "="*50 + "\n")
print(f"✅ '{TAG_TO_REMOVE}' removed and categories converted to string.")
display(final_events_df[['title', 'categories']].head())

In [ ]:
# Rename 'categories' to 'category'
final_events_df = final_events_df.rename(columns={'categories': 'category'})

display(final_events_df.head())

# Transformation logic - Part_4

In [ ]:
def extract_all_text(item):
    """
    Goes through every dictionary and list inside the JSON 
    until it finds 'textData'. No matter how deep it is.
    """
    text_parts = []

    # If it's a list, check every item in the list
    if isinstance(item, list):
        for sub_item in item:
            text_parts.append(extract_all_text(sub_item))

    # If it's a dictionary, look for textData OR more nodes
    elif isinstance(item, dict):
        # FOUND THE GOAL: Grab the text
        if 'textData' in item and 'text' in item['textData']:
            text_parts.append(item['textData']['text'])
        
        # ADD NEWLINES: If it's a paragraph or list item, add a break
        if item.get('type') in ['PARAGRAPH', 'LIST_ITEM', 'HEADING']:
            text_parts.append("\n")

        # DRILL DEEPER: Check every key in the dictionary for more lists/dicts
        for key, value in item.items():
            if isinstance(value, (list, dict)):
                text_parts.append(extract_all_text(value))

    return "".join(text_parts)

def clean_description(raw_nodes):
    """Entry point for the DataFrame apply."""
    if not raw_nodes:
        return ""
    full_text = extract_all_text(raw_nodes)
    
    # Clean up formatting: 
    # Fix multiple newlines
    text = re.sub(r'\n\s*\n+', '\n\n', full_text)
    # Fix spaces that might have been added between bold/normal text
    text = re.sub(r' +', ' ', text)
    return text.strip()

# Apply to your confirmed column 
final_events_df['descriptionText'] = events_df['description.nodes'].apply(clean_description)

print("✅ Description extraction should now capture the lists and nested paragraphs!")
display(final_events_df[['title', 'descriptionText']].head(2))

In [ ]:
# Update final columns to include your new description
final_columns = ['id', 'title', 'date', 'location', 'category', 'eventGuests', 'descriptionText', 'url']
final_events_df = final_events_df[final_columns]

print("✅ Description reconstructed.")
display(final_events_df.head(3))

# Export to JSON 

In [ ]:
# Keeping Hungarian characters safe with ensure_ascii=False
final_events_df.to_json(OUTPUT_FILE, orient='records', force_ascii=False, indent=4)
print(f"🚀 Data successfully exported to {OUTPUT_FILE}")